# 02 — Hillstrom: randomization check and ATE (Day 1)

Notebook 01 described the data. This one asks the causal question: **was the
e-mail assignment random, and how big is the average effect?** Both answers are
prerequisites for uplift modelling and both are baselines the models must beat.

Vocabulary lives in [`docs/concepts.md`](../../docs/concepts.md) — this notebook
introduces each idea briefly and links back rather than re-deriving it.

| Step | Question | Tool |
| --- | --- | --- |
| Arm choice | Which e-mail arm is worth modelling? | observed lift per arm |
| Randomization | Are treated and control comparable? | standardized mean differences |
| Effect size | What is the average effect? | ATE with 95% CI |

## 0. Setup

In [ ]:
import pathlib
import sys

ROOT = pathlib.Path.cwd().resolve()
for _ in range(8):
    if (ROOT / "pyproject.toml").exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd

from packages.causal.diagnostics import ate_table, smd_table
from packages.datasets.hillstrom import (
    FEATURES,
    OUTCOMES,
    arm_summary,
    fetch_hillstrom,
    load_hillstrom_raw,
)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

raw = load_hillstrom_raw()
print("repo root:", ROOT)

## 1. Choose the binary arm

An **arm** is one branch of the experiment; a **binary** experiment keeps one
e-mail arm and drops the other so the estimand is a clean two-group comparison
(see [concepts §1](../../docs/concepts.md)). Hillstrom has two e-mail arms, so
pick the one with the larger observed lift — with a reason, not a hunch.

In [ ]:
arm_summary(raw).round(4)

**Decision: `mens`.** Mens beats Womens on every outcome — `visit` 18.3% vs 15.1%
(vs 10.6% control), and more conversions — and it has the larger follow-up
conversion base, so its metrics will be less noisy. Nothing here argues for
Womens.

Set below; change it only with a recorded reason.

In [ ]:
ARM = "mens"          # "mens" or "womens"
df = fetch_hillstrom(arm=ARM)
print(df["segment"].value_counts().to_string())
print("\ntreated share:", round(df["treatment"].mean(), 4))
print("rows:", len(df), " conversions:", int(df["conversion"].sum()))
df.head()

### What is the `treatment` column?

The raw CSV has **no `treatment` column**. It only has `segment`, which holds three
text labels. Models need numbers, so `fetch_hillstrom(arm="mens")` builds one:

| `segment` (from the file) | | `treatment` (added by the loader) |
| --- | --- | --- |
| `Mens E-Mail` | → | `1` — got the e-mail |
| `No E-Mail` | → | `0` — got nothing |
| `Womens E-Mail` | → | *(row dropped)* |

In plain words: **`treatment` is a yes/no flag — 1 means "this customer received
the Mens e-mail", 0 means "they received no e-mail".**

It adds no new information. It is just the `segment` label rewritten as a number,
because the SMD table and the ATE need to group rows into "e-mailed" vs
"not e-mailed" and you cannot do arithmetic on the text `"Mens E-Mail"`.

Two consequences worth remembering:

- It exists only for a **binary** arm. With `arm="multi"` there is no `treatment`
  column — three arms cannot collapse into one 0/1 flag.
- Because it is derived from `segment`, it can never be a model feature — that
  would hand the model the answer. It is listed in `LEAKY_COLUMNS`.

## 2. Randomization check — standardized mean differences

Randomization means treatment is assigned independently of how a customer would
respond, so the two groups should look statistically identical *before* the
e-mail. **SMD** measures the gap on each feature in pooled-spread units so
different features are comparable; the convention here flags `|SMD| > 0.1`
([concepts §2](../../docs/concepts.md)).

Categorical features enter one-vs-rest (e.g. `zip_code=Urban`).

In [ ]:
smd = smd_table(df, FEATURES)
print("flagged (|SMD| > 0.1):", int(smd["flagged"].sum()), "of", len(smd))
print("largest |SMD|:", round(smd["abs_smd"].max(), 4))
smd.round(4)

**Finding.** Zero of 11 comparisons flagged; the largest `|SMD|` is far below
0.1. Treated and control are balanced on every pre-treatment feature, which is
what random assignment predicts.

**TODO [YOU]** — record your reading:

- Does the *pattern* of small SMDs look like noise (mixed signs, no structure),
  rather than a systematic difference?
- A gap this small is still detectable: at 21k per arm, an SMD of just 0.03
  (a 1.5-point share gap, or about $8 of mean history) is already *statistically*
  significant, p ~ 0.002. It is plainly too small to matter. Why does the
  rule of thumb use a fixed 0.1 instead of a p-value?

## 3. Average treatment effect (with 95% CI)

The **ATE** is mean(treated) − mean(control) per outcome; the 95% interval says
where the true average effect plausibly sits. Binary outcomes use the
two-proportion z interval, `spend` uses Welch's t interval
([concepts §2](../../docs/concepts.md)).

This is the number **every uplift model must beat**. Uniformly contacting everyone
buys exactly the ATE; random targeting scores ~0.

In [ ]:
ate = ate_table(df, OUTCOMES)
ate.round(4)

**TODO [YOU]** — fill in and interpret:

| Outcome | ATE | 95% CI | Distinguishable from 0? |
| --- | --- | --- | --- |
| `visit` | | | |
| `conversion` | | | |
| `spend` | | | |

- In words, what does the `visit` ATE mean for 1,000 contacted customers?
- Does `visit` move more than `conversion`, as expected? Why would that be?
- `spend`'s ATE is an average over a 99.1%-zero distribution (notebook 01 §5) —
  why is the interval wide relative to the effect, and what does that imply for
  profit estimates?

## 4. Profit assumptions

The ATE gives incremental *outcomes*; turning them into money needs assumptions.
They are stated in `configs/business_assumptions.yaml` and the README, never
buried in code:

| Assumption | Value |
| --- | --- |
| Reporting currency | USD (fixed — Hillstrom spend is already USD) |
| Gross margin on incremental spend | **TODO [YOU]** |
| Cost per contact (e-mail) | **TODO [YOU]** |
| Value per conversion (Criteo only) | **TODO [YOU]** |

Formula: `profit(k) = v · Δconv(k) − c · n(k)`, with `v` and `c` in the same
currency. See [concepts — profit formula](../../docs/concepts.md).

## 5. Handoff to Day 2

- [ ] SMD table reviewed; the "balanced, nothing flagged" verdict recorded
- [ ] ATE table interpreted in words, including the `visit` vs `conversion` gap
- [ ] Binary arm chosen and justified (`mens`, §1)
- [ ] Profit assumptions confirmed in `configs/business_assumptions.yaml` + README

Then Day 2: a response baseline versus S-/T-/X-learners, evaluated with Qini
curves — the first test of whether modelling heterogeneity beats the ATE.